In [ ]:
from typing import TypedDict

class Contact(TypedDict):
    name:str
    email:str
    phone:str
    
def send_email(contact:Contact) ->None:
    print(f"Sending email to {contact['name']} at {contact['email']}")

contact_info:Contact={
    'name':"Lilei",
    "email":"lilei@qq.com",
    "phone":"15555123123",
    "aaa":'fdasfasdfs'
}

send_email(contact_info)

In [ ]:
from langgraph.graph import StateGraph
from typing_extensions import TypedDict

class InputState(TypedDict):
    question:str
    
class OutputState(TypedDict):
    answer:str
class OverallState(InputState,OutputState):
    pass

builder=StateGraph(OverallState,input=InputState,output=OutputState)

def agent_node(state:InputState):
    print("我是一个AI agent")
    return

def action_node(state:InputState):
    print("我是一个执行者")
    return {'answer':f"我接收到的问题是 {state['question']},我现在执行成功了"}



In [ ]:
builder.add_node('agent_node',agent_node)
builder.add_node('action_node',action_node)



In [ ]:
from langgraph.graph import START,END

builder.add_edge(START,'agent_node')
builder.add_edge('agent_node','action_node')
builder.add_edge('action_node',END)



In [ ]:
graph=builder.compile()

In [ ]:
graph.invoke({"question":"哈喽,你好"})

In [ ]:
graph.invoke({'question':'今天的天气怎么样'})

In [ ]:
from langgraph.graph import StateGraph,START,END
from typing_extensions import TypedDict

class InputState(TypedDict):
    question:str
    
class OutputState(TypedDict):
    answer:str

class OverallState(InputState,OutputState):
    pass




In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()  # 加载.env文件里的变量
print(os.getenv("DEEPSEEK_API_KEY"))  # 现在可以正常读取了

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

import getpass




def llm_node(state:InputState):
    message=[
        ('system',"你是一位乐于助人的智能小助理"),
        ('human',state['question'])
    ]
    
    llm = ChatOpenAI(
        model="deepseek-chat",  # 使用的模型名称，目前官方推荐用 'deepseek-chat'
        api_key=os.getenv("DEEPSEEK_API_KEY"),  # 你的 DeepSeek API Key
        base_url="https://api.deepseek.com/v1",  # DeepSeek API 地址
        temperature=0,
    )
    
    response=llm.invoke(message)
    return {'answer':response.content}

In [ ]:
builder=StateGraph(OverallState,input=InputState,output=OutputState)

builder.add_node('llm_node',llm_node)
builder.add_edge(START,'llm_node')
builder.add_edge('llm_node',END)

graph=builder.compile()

In [ ]:
graph.invoke({"question":'你好， 我用来测试'})

In [ ]:
final_answer=graph.invoke({'question':'你好，我用来测试'})
print(final_answer['answer'])

In [ ]:
final_answer=graph.invoke({'question':'你好，请你详细的介绍一下你自己'})
print(final_answer['answer'])

In [ ]:
from langgraph.graph import StateGraph,START,END
from typing_extensions import TypedDict

class InputState(TypedDict):
    question:str
    llm_answer:str
    
class OutputState(TypedDict):
    answer:str

class OverallState(InputState,OutputState):
    pass

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate


def llm_node(state:InputState):
    message=[
        ('system',"你是一位乐于助人的智能小助理"),
        ('human',state['question'])
    ]
    
    llm = ChatOpenAI(
        model="deepseek-chat",  # 使用的模型名称，目前官方推荐用 'deepseek-chat'
        api_key=os.getenv("DEEPSEEK_API_KEY"),  # 你的 DeepSeek API Key
        base_url="https://api.deepseek.com/v1",  # DeepSeek API 地址
        temperature=0,
    )
    
    response=llm.invoke(message)
    return {'llm_answer':response.content}


def action_node(state:InputState):
    messages=[
        ('system',"无论你接收到什么语言的文本，请翻译成英语"),
        ('human',state['llm_answer'])
    ]
    
    llm = ChatOpenAI(
            model="deepseek-chat",  # 使用的模型名称，目前官方推荐用 'deepseek-chat'
            api_key=os.getenv("DEEPSEEK_API_KEY"),  # 你的 DeepSeek API Key
            base_url="https://api.deepseek.com/v1",  # DeepSeek API 地址
            temperature=0,
        )
    response=llm.invoke(messages)
    return {'answer':response.content}

In [ ]:
builder=StateGraph(OverallState,input=InputState,output=OutputState)

builder.add_node('action_node',action_node)
builder.add_node('llm_node',llm_node)

builder.add_edge(START,'llm_node')
builder.add_edge('llm_node','action_node')
builder.add_edge('action_node',END)

graph=builder.compile()

In [ ]:
graph.invoke({'question':'你好，你是谁'})